# PANN 25x3s – Save Embeddings

This notebook extracts PANN embeddings once from 25 random 3-second audio segments per track and saves them to one file. The model notebooks can then load this file directly, which avoids repeating the expensive PANN inference step.

In [ ]:
import sys
from pathlib import Path

current_path = Path.cwd()

for parent in [current_path] + list(current_path.parents):
    if (parent / "src").exists():
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError("Could not find project root containing 'src' folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import torch
import sys
import os
from panns_inference import AudioTagging
import librosa
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
from src.config import (
    PANN_AUDIO_DIR,
    PANN_METADATA_CSV,
    PANN_CHECKPOINT_PATH,
    PANN_3S25_EMBEDDING_FILE,
    PANN_SAMPLE_RATE,
    PANN_SEGMENT_LENGTH_SECONDS,
    PANN_N_SEGMENTS_PER_TRACK,
    EXCLUDED_GENRES,
    TOP_K_GENRES as CONFIG_TOP_K_GENRES,
    RANDOM_STATE as CONFIG_RANDOM_STATE
)


In [ ]:
# ============================================================
# CONFIGURATION
# Central place to control all experiment parameters
# ============================================================

# Paths
DATA_DIR = str(PANN_AUDIO_DIR)
CSV_FILE = str(PANN_METADATA_CSV)
CHECKPOINT_PATH = str(PANN_CHECKPOINT_PATH)
EMBEDDING_OUTPUT_FILE = str(PANN_3S25_EMBEDDING_FILE)
MODEL_NAME = "PANN_25x3s_embeddings"

TOP_K_GENRES = CONFIG_TOP_K_GENRES                      # Keep only the top K most frequent genres
EXCLUDE_GENRES = EXCLUDED_GENRES # Exclude specific genres 
SAMPLES_PER_GENRE = 533               # Balance dataset per genre before segment extraction
TEST_SPLIT_RATIO = 0.2                # Train vs Rest split on track level
VAL_SPLIT_RATIO = 0.5                 # Validation vs Test split within Rest
RANDOM_STATE = CONFIG_RANDOM_STATE                     # Reproducibility

# Audio / segment sampling
SEGMENT_LENGTH_SECONDS = PANN_SEGMENT_LENGTH_SECONDS            # Length of every random segment
N_RANDOM_SEGMENTS = PANN_N_SEGMENTS_PER_TRACK                # Number of random 3-second samples per track
SAMPLING_RATE = PANN_SAMPLE_RATE                 # Target sampling rate required by PANN

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


In [11]:
# ---------------------------------------------------------
# LOAD AND FILTER DATA
# ---------------------------------------------------------

# Load metadata (track_id as string)
metadata = pd.read_csv(CSV_FILE, dtype={'track_id': str})
metadata = metadata[["track_id", "track_genre_top"]]

# Scan physical audio files on disk
existing_files = set(os.listdir(DATA_DIR))
print(f"Files found on disk: {len(existing_files)}")

# Build padded filenames (e.g., 123 → 000123.wav)
metadata['filename'] = metadata['track_id'].str.zfill(6) + ".wav"

# Keep only tracks that physically exist
df_filtered = metadata[metadata['filename'].isin(existing_files)].copy()

# Remove tracks without genre labels
df_filtered = df_filtered.dropna(subset=['track_genre_top'])
print(f"After filtering existing files and removing NaNs: {len(df_filtered)}")

Files found on disk: 103871
After filtering existing files and removing NaNs: 48148


In [12]:
# Exclude unwanted genres
if EXCLUDE_GENRES:
    df_filtered = df_filtered[~df_filtered['track_genre_top'].isin(EXCLUDE_GENRES)]
    print(f"Excluded genres: {EXCLUDE_GENRES}")

# Keep only Top-K genres
if TOP_K_GENRES is not None:
    genre_counts = df_filtered['track_genre_top'].value_counts()
    top_genres = genre_counts.head(TOP_K_GENRES).index.tolist()
    
    df_combined = df_filtered[df_filtered['track_genre_top'].isin(top_genres)].reset_index(drop=True)
    print(f"Filtered to Top {TOP_K_GENRES} genres: {top_genres}")
else:
    df_combined = df_filtered.reset_index(drop=True)

# Add full file paths
df_combined['file_path'] = df_combined['filename'].apply(
    lambda x: os.path.join(DATA_DIR, x)
)

# Balance dataset (fixed samples per genre)
if SAMPLES_PER_GENRE is not None:
    sampled_rows = []
    for genre, group in df_combined.groupby("track_genre_top"):
        if len(group) >= SAMPLES_PER_GENRE:
            sampled_rows.append(group.sample(n=SAMPLES_PER_GENRE, random_state=RANDOM_STATE))
        else:
            sampled_rows.append(group)
    df_final = pd.concat(sampled_rows).reset_index(drop=True)
else:
    df_final = df_combined

audio_files = df_final['file_path'].tolist()
labels = df_final['track_genre_top'].tolist()

print(f"\nFinal dataset size: {len(audio_files)} samples.")

Excluded genres: ['International', 'Instrumental', 'Experimental']
Filtered to Top 8 genres: ['Rock', 'Electronic', 'Hip-Hop', 'Folk', 'Pop', 'Classical', 'Jazz', 'Old-Time / Historic']

Final dataset size: 4264 samples.


In [13]:
# ---------------------------------------------------------
# LABEL CLEANING BEFORE FEATURE EXTRACTION
# ---------------------------------------------------------

cleaned_labels = []
cleaned_audio_files = []

for label, file_path in zip(labels, audio_files):
    # Keep only valid string labels (exclude NaN / floats)
    if isinstance(label, str):
        cleaned_labels.append(label)
        cleaned_audio_files.append(file_path)

labels = cleaned_labels
audio_files = cleaned_audio_files

if not labels:
    print("ERROR: No valid samples with string labels found.")
    sys.exit()

# Create label mappings
label_names = sorted(set(labels))
label_to_idx = {l: i for i, l in enumerate(label_names)}
idx_to_label = {i: l for l, i in label_to_idx.items()}

In [14]:
# ---------------------------------------------------------
# PANNs FEATURE EXTRACTION WITH 25 RANDOM 3-SECOND SEGMENTS
# ---------------------------------------------------------

at = AudioTagging(checkpoint_path=CHECKPOINT_PATH, device=DEVICE)


def sample_random_audio_segments(audio, sr=16000, segment_sec=3, n_segments=25, rng=None):
    """
    Draw n_segments random fixed-length segments from one audio signal.
    If the file is shorter than the target length, it is padded with zeros.

    Returns
    -------
    list[np.ndarray]
        List of mono audio segments, each with shape (segment_len,).
    """
    if rng is None:
        rng = np.random.default_rng(RANDOM_STATE)

    segment_len = int(segment_sec * sr)

    if len(audio) < segment_len:
        audio = np.pad(audio, (0, segment_len - len(audio)), mode="constant")

    max_start = len(audio) - segment_len
    starts = rng.integers(0, max_start + 1, size=n_segments) if max_start > 0 else np.zeros(n_segments, dtype=int)

    return [audio[start:start + segment_len] for start in starts]


def build_pann_random_segment_dataset(df, at, sr=16000, segment_sec=3, n_segments=25, random_state=42):
    """
    Builds a segment-level PANN embedding dataset.
    Each original track becomes n_segments independent 3-second samples.
    """
    embeddings = []
    y_labels = []
    track_ids = []
    segment_indices = []
    file_names = []
    skipped = 0

    rng = np.random.default_rng(random_state)

    for i, row in df.reset_index(drop=True).iterrows():
        path = row["file_path"]
        label = row["track_genre_top"]
        track_id = row["track_id"]
        file_name = row.get("filename", os.path.basename(path))

        try:
            audio, _ = librosa.load(path, sr=sr, mono=True)
            segments = sample_random_audio_segments(
                audio,
                sr=sr,
                segment_sec=segment_sec,
                n_segments=n_segments,
                rng=rng
            )

            for segment_idx, segment in enumerate(segments):
                pann_input = segment[None, :]  # PANN expects shape: (batch_size, samples)
                _, embedding = at.inference(pann_input)

                embeddings.append(np.asarray(embedding).reshape(-1))
                y_labels.append(label)
                track_ids.append(track_id)
                segment_indices.append(segment_idx)
                file_names.append(file_name)

        except Exception as e:
            skipped += 1
            print(f"Skipped {path}: {e}")

        if i % 200 == 0:
            print(f"Processed tracks: {i}/{len(df)} | Segment samples: {len(y_labels)} | Skipped tracks: {skipped}")

    return (
        np.vstack(embeddings),
        np.array(y_labels),
        np.array(track_ids),
        np.array(segment_indices),
        np.array(file_names),
        skipped
    )


# Split tracks first, then extract random segments.
# This prevents segments from the same track appearing in different splits.
train_df, temp_df = train_test_split(
    df_final,
    test_size=TEST_SPLIT_RATIO,
    random_state=RANDOM_STATE,
    stratify=df_final["track_genre_top"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=VAL_SPLIT_RATIO,
    random_state=RANDOM_STATE,
    stratify=temp_df["track_genre_top"]
)

print(f"Track split sizes -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

split_dfs = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}

split_features = {}
skipped_by_split = {}

for split_name, split_df in split_dfs.items():
    print("=" * 80)
    print(f"Extracting PANN embeddings for {split_name}: {len(split_df)} tracks")

    X_split, y_split_labels, track_ids_split, segment_ids_split, file_names_split, skipped = build_pann_random_segment_dataset(
        df=split_df,
        at=at,
        sr=SAMPLING_RATE,
        segment_sec=SEGMENT_LENGTH_SECONDS,
        n_segments=N_RANDOM_SEGMENTS,
        random_state=RANDOM_STATE
    )

    split_features[split_name] = {
        "X": X_split,
        "y_labels": y_split_labels,
        "track_ids": track_ids_split,
        "segment_ids": segment_ids_split,
        "file_names": file_names_split
    }
    skipped_by_split[split_name] = skipped

    print(f"{split_name}: {X_split.shape[0]} segment samples, embedding shape: {X_split.shape}")

# Build arrays used by the following modelling cells
X_train = split_features["train"]["X"]
X_val = split_features["val"]["X"]
X_test = split_features["test"]["X"]

y_train_labels = split_features["train"]["y_labels"]
y_val_labels = split_features["val"]["y_labels"]
y_test_labels = split_features["test"]["y_labels"]

label_encoder = LabelEncoder()
label_encoder.fit(np.concatenate([y_train_labels, y_val_labels, y_test_labels]))

label_names = list(label_encoder.classes_)
label_to_idx = {label: idx for idx, label in enumerate(label_names)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

y_train = label_encoder.transform(y_train_labels)
y_val = label_encoder.transform(y_val_labels)
y_test = label_encoder.transform(y_test_labels)

track_ids_test = split_features["test"]["track_ids"]
segment_ids_test = split_features["test"]["segment_ids"]
labels_test = y_test_labels

# Compatibility variables for older cells
embeddings = np.vstack([X_train, X_val, X_test])
y_true_numerical = np.concatenate([y_train, y_val, y_test])
track_ids_filtered = np.concatenate([
    split_features["train"]["track_ids"],
    split_features["val"]["track_ids"],
    split_features["test"]["track_ids"]
])

print("\n--- Feature extraction completed ---")
print(f"Train segments: {len(X_train)} | Val segments: {len(X_val)} | Test segments: {len(X_test)}")
print(f"Expected per track: {N_RANDOM_SEGMENTS} random segments of {SEGMENT_LENGTH_SECONDS} seconds")


Checkpoint path: J:/Documents/FH/Models/audioset_tagging_cnn-master/pytorch/Cnn14_mAP=0.431.pth


j:\Documents\FH\Git Repo Processing\MA_FMA_Dataprocessing\venv\Lib\site-packages\panns_inference\inference.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = 

GPU number: 1
Track split sizes -> Train: 3411, Val: 426, Test: 427
Extracting PANN embeddings for train: 3411 tracks
Processed tracks: 0/3411 | Segment samples: 25 | Skipped tracks: 0
Processed tracks: 200/3411 | Segment samples: 5025 | Skipped tracks: 0
Processed tracks: 400/3411 | Segment samples: 10025 | Skipped tracks: 0
Processed tracks: 600/3411 | Segment samples: 15025 | Skipped tracks: 0
Processed tracks: 800/3411 | Segment samples: 20025 | Skipped tracks: 0
Processed tracks: 1000/3411 | Segment samples: 25025 | Skipped tracks: 0
Processed tracks: 1200/3411 | Segment samples: 30025 | Skipped tracks: 0
Processed tracks: 1400/3411 | Segment samples: 35025 | Skipped tracks: 0
Processed tracks: 1600/3411 | Segment samples: 40025 | Skipped tracks: 0
Processed tracks: 1800/3411 | Segment samples: 45025 | Skipped tracks: 0
Processed tracks: 2000/3411 | Segment samples: 50025 | Skipped tracks: 0
Processed tracks: 2200/3411 | Segment samples: 55025 | Skipped tracks: 0
Processed tracks:

In [15]:
# ---------------------------------------------------------
# SAVE EXTRACTED PANN EMBEDDINGS
# ---------------------------------------------------------
# The saved file contains all arrays and metadata needed by the four model notebooks.
# After running this notebook once, Logistic Regression, SVM, Random Forest and MLP can load this file directly.

os.makedirs(os.path.dirname(EMBEDDING_OUTPUT_FILE), exist_ok=True)

pann_embedding_data = {
    "config": {
        "data_dir": DATA_DIR,
        "csv_file": CSV_FILE,
        "top_k_genres": TOP_K_GENRES,
        "exclude_genres": EXCLUDE_GENRES,
        "samples_per_genre": SAMPLES_PER_GENRE,
        "test_split_ratio": TEST_SPLIT_RATIO,
        "val_split_ratio": VAL_SPLIT_RATIO,
        "random_state": RANDOM_STATE,
        "segment_length_seconds": SEGMENT_LENGTH_SECONDS,
        "n_random_segments": N_RANDOM_SEGMENTS,
        "sampling_rate": SAMPLING_RATE,
    },
    "split_features": split_features,
    "skipped_by_split": skipped_by_split,
    "train_df": train_df.reset_index(drop=True),
    "val_df": val_df.reset_index(drop=True),
    "test_df": test_df.reset_index(drop=True),
    "label_classes": label_encoder.classes_,
}

joblib.dump(pann_embedding_data, EMBEDDING_OUTPUT_FILE, compress=3)

print(f"[SAVED] PANN embeddings written to: {EMBEDDING_OUTPUT_FILE}")
print(f"Train embeddings: {split_features['train']['X'].shape}")
print(f"Val embeddings:   {split_features['val']['X'].shape}")
print(f"Test embeddings:  {split_features['test']['X'].shape}")
print("Use this file as EMBEDDING_FILE in the model notebooks.")


[SAVED] PANN embeddings written to: J:/Documents/FH/Git Repo Processing/MA_FMA_Dataprocessing/Models_final/PANN_25x3s_embeddings.joblib
Train embeddings: (85275, 2048)
Val embeddings:   (10650, 2048)
Test embeddings:  (10675, 2048)
Use this file as EMBEDDING_FILE in the model notebooks.
